In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

keys = {
    "Gemini": os.getenv("GEMINI_API_KEY"),
    "Hugging Face": os.getenv("HF_TOKEN"),
    "Groq": os.getenv("GROQ_API_KEY"),
    "OpenRouter": os.getenv("OPENROUTER_API_KEY")
}

for name, key in keys.items():
    if key:
        print(f"{name}: ✅ Key loaded")
        print(f"Starts with: {key[:6]}...")
    else:
        print(f"{name}: ❌ Key NOT loaded")

Gemini: ✅ Key loaded
Starts with: AQ.Ab8...
Hugging Face: ✅ Key loaded
Starts with: hf_Erw...
Groq: ✅ Key loaded
Starts with: gsk_gd...
OpenRouter: ✅ Key loaded
Starts with: sk-or-...


In [4]:
import os
import requests
import gradio as gr
from dotenv import load_dotenv

load_dotenv()

C:\Users\SATISFY\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [5]:
!pip install gradio python-dotenv requests

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\SATISFY\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
!pip install gradio python-dotenv requests

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\SATISFY\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
print("OpenRouter model:", os.getenv("OPENROUTER_MODEL"))

OpenRouter model: openai/gpt-oss-20b:free


In [9]:
load_dotenv()

API_KEY = os.getenv("OPENROUTER_API_KEY")
MODEL = os.getenv("OPENROUTER_MODEL")

print("API key loaded:", API_KEY is not None)
print("Model loaded:", MODEL is not None)

API key loaded: True
Model loaded: True


In [10]:
def get_ai_response(messages):

    if not API_KEY:
        return "Error: OpenRouter API key was not found in your .env file."

    if not MODEL:
        return "Error: OpenRouter model was not found in your .env file."

    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": MODEL,
        "messages": messages
    }

    try:
        response = requests.post(
            url,
            headers=headers,
            json=data,
            timeout=60
        )

        response.raise_for_status()

        result = response.json()

        return result["choices"][0]["message"]["content"]

    except requests.exceptions.HTTPError:
        return f"API Error: {response.status_code} - {response.text}"

    except Exception as e:
        return f"Error: {str(e)}"

In [11]:
SYSTEM_PROMPT = """
You are the official AI customer service representative for
SUKKY'S DELIGHT, an international food restaurant.

Your personality should be warm, pleasant, welcoming, patient and genuinely
helpful. Speak to customers like a professional hospitality representative,
not like a robotic AI.

ABOUT SUKKY'S DELIGHT:
SUKKY'S DELIGHT is an international food restaurant offering a variety of
dishes inspired by different cuisines. The restaurant also has a rooftop
space that customers can reserve.

YOUR RESPONSIBILITIES:

1. CUSTOMER EMAIL RESPONSES
Respond naturally to customer emails and messages.

Always:
- Be polite and welcoming.
- Show empathy when appropriate.
- Thank customers when appropriate.
- Keep responses clear and easy to read.
- Maintain a professional but friendly personality.

2. FOOD INFORMATION
Help customers learn about available dishes, including:
- Dish name
- Cuisine/origin
- Ingredients
- Flavor
- Spice level
- Dietary information

Never invent a dish, ingredient, price or availability.

3. FOOD RECOMMENDATIONS
Help customers choose meals based on preferences such as:
- Spicy or mild
- Vegetarian
- Vegan
- Light or filling
- New to international cuisine

Only recommend dishes that are actually available.

4. ALLERGIES AND DIETARY REQUIREMENTS
Never guarantee that a dish is safe for an allergy unless verified
information has been provided.

If allergen information is unavailable, tell the customer that it must
be confirmed by the restaurant before ordering.

5. ORDERS AND DELIVERY
Help customers with general order questions.

Never invent:
- Order status
- Delivery time
- Refunds
- Replacements
- Discounts
- Order numbers

Ask for the appropriate information when necessary.

6. CUSTOMER COMPLAINTS AND ESCALATION
If a customer is angry, has a serious complaint, was charged incorrectly,
received the wrong order, has a missing order, or requests something that
requires human intervention:

- Remain calm and empathetic.
- Apologize when appropriate.
- Acknowledge the customer's concern.
- Explain that the matter can be escalated to the appropriate team.
- Do not promise a refund, replacement or compensation unless authorized.

7. ROOFTOP RESERVATIONS
SUKKY'S DELIGHT has a rooftop space that customers can request to reserve.

If a customer wants to make a reservation, collect:
- Name
- Date
- Preferred time
- Number of guests
- Any special request

Do NOT claim that a reservation is confirmed unless the booking system
actually confirms it.

If availability cannot be checked, tell the customer that the request
will need to be confirmed by the restaurant.

8. RESTAURANT LOCATION
If a customer asks for the restaurant location, provide the official
restaurant location supplied by the restaurant.

Never invent an address.

9. PRIVACY AND SECURITY
Never request:
- Passwords
- PINs
- API keys
- Full debit/credit card numbers
- Banking passwords
- Authentication codes

10. COMMUNICATION STYLE
Be:
- Pleasant
- Warm
- Respectful
- Patient
- Helpful
- Professional

Avoid:
- Robotic language
- Arguments
- Blaming customers
- Unnecessary long responses
- Making unsupported claims

Always make the customer feel welcomed at SUKKY'S DELIGHT.

When appropriate, end responses with a friendly closing such as:

"Thank you for choosing SUKKY'S DELIGHT. We look forward to serving you!"
"""

In [12]:
RESTAURANT_INFO = {
    "name": "SUKKY'S DELIGHT",

    "address": " Basin",

    "city":"MANCHESTER",

    "rooftop": "Our rooftop is available for reservations.",

    "phone": "4447885636332",

    "email": "info@delight.com"
}

In [13]:
def chatbot(message, history):

    if not message.strip():
        return history

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ]

    # Add previous conversation
    for item in history:
        if isinstance(item, dict):
            messages.append({
                "role": item["role"],
                "content": item["content"]
            })

        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_message, assistant_message = item

            if user_message:
                messages.append({
                    "role": "user",
                    "content": user_message
                })

            if assistant_message:
                messages.append({
                    "role": "assistant",
                    "content": assistant_message
                })

    # Add the new customer message
    messages.append({
        "role": "user",
        "content": message
    })

    response = get_ai_response(messages)

    # Return the new conversation in Gradio's message format
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]

    return history

In [14]:
print("API key loaded:", API_KEY is not None)
print("Model:", MODEL)

test = get_ai_response([
    {
        "role": "user",
        "content": "Hello"
    }
])

print(test)

API key loaded: True
Model: openai/gpt-oss-20b:free
API Error: 404 - {"error":{"message":"This model is unavailable for free. The paid version is available now - use this slug instead: openai/gpt-oss-20b","code":404},"user_id":"user_3A0Aw6gLSFsOE3YCTxYTwqMGRJY"}


In [15]:
import gradio as gr

In [16]:
with gr.Blocks(title="Sukkys Delight Customer Service") as app:

    gr.Markdown(
        """
        # 🌍 International Food Customer Service

        ### 📧 AI Customer Email Responder

        Send a customer's email or message and the AI will generate a
        professional, friendly and helpful customer-service response.
        """
    )

    chatbot_ui = gr.Chatbot(
        label="Customer Service Chat",
        height=500
    )

    with gr.Row():

        message_box = gr.Textbox(
            label="Customer Message / Email",
            placeholder="Paste or type the customer's email here...",
            lines=4,
            scale=8
        )

        send_button = gr.Button(
            "Send",
            variant="primary",
            scale=1
        )

    clear_button = gr.Button("🗑️ Clear Chat")

    send_button.click(
        chatbot,
        inputs=[message_box, chatbot_ui],
        outputs=chatbot_ui
    ).then(
        lambda: "",
        outputs=message_box
    )

    message_box.submit(
        chatbot,
        inputs=[message_box, chatbot_ui],
        outputs=chatbot_ui
    ).then(
        lambda: "",
        outputs=message_box
    )

    clear_button.click(
        lambda: [],
        outputs=chatbot_ui
    )


app.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
